## Introduction - Why LangChain Changed the Way I Build with LLMs

When I first started experimenting with large language models like GPT, I was amazed by their capabilities. But I quickly hit a wall: every useful task - like answering questions from my own documents, remembering past conversations, or using a calculator — required messy glue code. That's when I discovered LangChain.

LangChain isn't just another Python library. It's a framework that turns LLMs from fancy text generators into actual application components. It solves problems like prompt management, chaining multiple calls, adding memory, and connecting external tools. Think of it as the "plumbing" that makes LLMs useful in real software.

In this notebook, I'll walk through LangChain's core pieces, show you working code, and share three real use cases. By the end, you'll know not just *how* to use LangChain, but *when* and *why*.

In [1]:
!pip install langchain langchain-classic langchain-openai langchain-community python-dotenv openai -q

print("✅ All dependencies installed!")

✅ All dependencies installed!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Bindu\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Check if API key is already set
if not os.environ.get("OPENAI_API_KEY"):
    print("🔑 Please enter your OpenAI API key:")
    os.environ["OPENAI_API_KEY"] = getpass("API Key: ")
    print("✅ API key set!")
else:
    print("✅ API key found!")

# Import all necessary modules for LangChain 1.0+
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import Tool
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain, LLMChain

# For agents - these are now in langchain_community
from langchain_community.agent_toolkits import create_react_agent
from langchain_community.tools import load_tools
from langchain.agents import AgentExecutor
from langchain.agents.format_scratchpad import format_to_openai_function_messages
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser

# Alternative simple agent import (works in 1.0+)
from langchain.agents import create_openai_tools_agent, AgentExecutor

print("\n✅ All imports successful!")
print(f"Python version: {os.sys.version}")
print("Note: Pydantic warning about Python 3.14 is harmless - code will still work")


What: A wrapper around an LLM API (OpenAI, HuggingFace, etc.).  
Why it exists:Standardizes input/output so you can swap models without rewriting code.

In [ ]:
# Initialize the LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Make a simple call
response = llm.invoke("Explain what LangChain is in one sentence.")

print("=" * 50)
print("BASIC LLM CALL EXAMPLE")
print("=" * 50)
print(f"Response: {response.content}")

### 2. Prompts and Prompt Templates

**What:** Reusable templates that inject variables into prompts.  
**Why:** Avoid hardcoding prompts; maintain them like code.

In [ ]:
# Create a prompt template with variables
template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {role}."),
    ("human", "Can you explain {topic} in a simple and easy way?")
])

# Format the template (without calling LLM)
formatted_prompt = template.format(
    role="physics",
    topic="quantum entanglement"
)

print("=" * 50)
print("PROMPT TEMPLATE EXAMPLE")
print("=" * 50)
print("Formatted prompt:")
print(formatted_prompt)

# Use it with LLM
chain = template | llm
result = chain.invoke({"role": "physics", "topic": "quantum entanglement"})

print("\n" + "-" * 50)
print("LLM Response:")
print(result.content)

### 3. Chains

**What:** Combine prompts + LLMs into a single callable object.  
**Why:** Move from one-off scripts to modular pipelines.

In [ ]:
# Using LLMChain
chain = LLMChain(llm=llm, prompt=template)
result = chain.run(role="history", topic="cold war")

print("=" * 50)
print("CHAIN EXAMPLE")
print("=" * 50)
print(f"Response: {result}")

### 4. Memory

**What:** Stores and retrieves past conversation turns.  
**Why:** Without memory, every LLM call is stateless (like talking to a goldfish).

In [ ]:
# Create memory
memory = ConversationBufferMemory(return_messages=True)

# Create conversation chain with memory
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

print("=" * 50)
print("MEMORY DEMO")
print("=" * 50)

# First message
response1 = conversation.predict(input="Hi, my name is Samir.")
print(f"User: Hi, my name is Samir.")
print(f"Assistant: {response1}")

# Second message - tests memory
response2 = conversation.predict(input="What's my name?")
print(f"\nUser: What's my name?")
print(f"Assistant: {response2}")

# Show memory contents
print("\n" + "-" * 30)
print("What memory stores:")
print(memory.buffer)

### 5. Tools & Agents

**What:** Tools are functions (e.g., search, calculator). Agents decide which tool to use and when.  
**Why:** LLMs alone can't do math or fetch live data. Agents bridge that gap.

In [ ]:
# Load tools (calculator)
tools = load_tools(["llm-math"], llm=llm)

# Initialize agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False,
    handle_parsing_errors=True
)

print("=" * 50)
print("AGENT WITH CALCULATOR TOOL")
print("=" * 50)

# Ask a math question - agent uses calculator
result = agent.run("What's 237 * 581 divided by 3?")
print(f"Question: What's 237 * 581 divided by 3?")
print(f"Answer: {result}")

In [ ]:
def research_assistant():
    """
    Complete research assistant with memory and search
    """
    # Create memory
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )
    
    # Try to use Wikipedia, fallback to mock
    try:
        from langchain_community.utilities import WikipediaAPIWrapper
        wikipedia = WikipediaAPIWrapper()
        
        tools = [
            Tool(
                name="Wikipedia",
                func=wikipedia.run,
                description="Search Wikipedia for information"
            )
        ]
        print("✅ Using real Wikipedia search")
    except:
        # Mock search function
        def mock_search(query):
            if "mars" in query.lower():
                return "Mars rovers are exploring new regions and sending high-quality images."
            elif "quantum" in query.lower():
                return "Quantum computing uses qubits that exist in multiple states."
            else:
                return f"Information about: {query}"
        
        tools = [
            Tool(
                name="Search",
                func=mock_search,
                description="Search for information"
            )
        ]
        print("⚠️ Using mock search (install langchain-community for real Wikipedia)")
    
    # Create agent
    agent = initialize_agent(
        tools=tools,
        llm=llm,
        agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
        memory=memory,
        verbose=False,
        max_iterations=3
    )
    
    return agent

# Create and test the assistant
assistant = research_assistant()

print("\n" + "=" * 50)
print("RESEARCH ASSISTANT DEMO")
print("=" * 50)

# Ask questions
questions = [
    "What's the latest news on Mars rovers?",
    "And how does that compare to earlier missions?"
]

for q in questions:
    print(f"\nUser: {q}")
    response = assistant.invoke({"input": q})
    print(f"Assistant: {response['output']}")

## Real-World Use Cases

### 1. Customer Support Bot for E-commerce Store

**Problem:** Repeated questions about order status, returns, and policies.  
**Solution:** LangChain agent with memory + tool to query order database.  
**Components:** Chat model, memory, custom tool (SQL query), prompt templates.

### 2. Meeting Transcript Summarizer

**Problem:** Hour-long calls produce too much text; teams miss decisions.  
**Solution:** Chain that splits transcript → summarizes each segment → extracts action items.  
**Components:** Document loader, LLMChain, output parser.

### 3. Personal Financial Advisor Bot

**Problem:** Users want to ask "How much did I spend on coffee last month?" without sharing bank data.  
**Solution:** Local-first agent using local LLM + tool to query local CSV.  
**Components:** Local LLM, custom tool, memory, vector store.

## Advantages and Limitations

### Advantages

| Aspect | Benefit |
|--------|---------|
| **Modularity** | Swap any piece (LLM, memory, prompt) without breaking the rest |
| **Rapid prototyping** | Built research assistant in under 30 minutes |
| **Integrations** | Dozens of document loaders, vector stores, and tools |

### Limitations

| Aspect | Challenge |
|--------|-----------|
| **Latency** | Each agent "thought" step means multiple LLM calls |
| **Debugging** | Hard to trace where failures occur |
| **Cost** | Chaining + memory + agents increases token usage |

### When NOT to use LangChain
- Single LLM call with no memory or tools
- Extremely latency-sensitive applications (<200ms)
- Budget-constrained projects where every token counts

## Conclusion — My Key Takeaways

Working with LangChain taught me that LLMs are like CPUs: powerful but useless without memory, I/O, and instructions. LangChain provides those missing pieces. I learned to:

- **Think in components** rather than scripts
- **Design for replaceability** (today's GPT-4 might be tomorrow's open-source model)
- **Understand the cost** of agent reasoning steps

### Future Scope
The LangChain ecosystem is evolving toward:
- **LangGraph** – Stateful, multi-agent workflows
- **LangSmith** – Debugging and monitoring
- **Multi-agent systems** – Specialized agents collaborating
